In [ ]:
!nvidia-smi

# Qwen 2.5 14B: 20-seed hardening

Model: Qwen/Qwen2.5-14B | [HF](https://huggingface.co/Qwen/Qwen2.5-14B)\
GPU: H100 SXM 80GB | 20 seeds | 350 ex/dim | Peak layer from v3 | Date: 2026-04-08

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

In [ ]:
!pip install transformers datasets scipy huggingface_hub -q

try:
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("Not on Colab or no HF_TOKEN secret")

In [ ]:
# Pre-download models (uses HF_TOKEN from env)
!hf download Qwen/Qwen2.5-14B


In [ ]:
import gc
import json

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, rankdata, spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
def load_wikitext(split='test', max_docs=None):
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split=split)
    docs, current = [], []
    for row in ds:
        text = row['text']
        if text.strip() == '' and current:
            docs.append('\n'.join(current)); current = []
            if max_docs and len(docs) >= max_docs: break
        elif text.strip(): current.append(text)
    if current: docs.append('\n'.join(current))
    return docs

def partial_spearman(x, y, covariates):
    rx, ry = rankdata(x), rankdata(y)
    rc = np.column_stack([rankdata(c) for c in covariates])
    rc = np.column_stack([rc, np.ones(len(rc))])
    coef_x = np.linalg.lstsq(rc, rx, rcond=None)[0]
    coef_y = np.linalg.lstsq(rc, ry, rcond=None)[0]
    r, p = pearsonr(rx - rc @ coef_x, ry - rc @ coef_y)
    return float(r), float(p)

def compute_loss_residuals(losses, max_softmax, activation_norm):
    X = np.column_stack([max_softmax, activation_norm, np.ones(len(losses))])
    beta, _, _, _ = np.linalg.lstsq(X, losses, rcond=None)
    return losses - X @ beta

def collect_layer_data(model, tokenizer, docs, layer, device, max_tokens=200000, max_length=512):
    model.eval()
    all_acts, all_losses, all_softmax, all_norms = [], [], [], []
    total = 0
    for doc in docs:
        if total >= max_tokens: break
        if not doc.strip(): continue
        tokens = tokenizer(doc, return_tensors='pt', truncation=True, max_length=max_length)
        input_ids = tokens['input_ids'].to(device)
        if input_ids.size(1) < 2: continue
        with torch.inference_mode():
            outputs = model(input_ids, output_hidden_states=True)
        h = outputs.hidden_states[layer + 1][0, :-1, :].cpu()
        logits = outputs.logits[0, :-1, :]
        labels = input_ids[0, 1:]
        losses = F.cross_entropy(logits, labels, reduction='none').cpu()
        sm = F.softmax(logits, dim=-1).max(dim=-1).values.cpu()
        norms = h.norm(dim=-1)
        all_acts.append(h); all_losses.append(losses); all_softmax.append(sm); all_norms.append(norms)
        total += h.size(0)
    print(f'    {total} positions from {len(all_acts)} documents')
    return {
        'activations': torch.cat(all_acts).float(),
        'losses': torch.cat(all_losses).float().numpy(),
        'max_softmax': torch.cat(all_softmax).float().numpy(),
        'activation_norm': torch.cat(all_norms).float().numpy(),
    }

def train_linear_binary(train_data, seed=42, epochs=20, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    acts = train_data['activations']
    residuals = compute_loss_residuals(
        train_data['losses'], train_data['max_softmax'], train_data['activation_norm'])
    targets = torch.from_numpy((residuals > 0).astype(np.float32))
    head = torch.nn.Linear(acts.size(1), 1)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(acts, targets)
    dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
    head.train()
    for _ in range(epochs):
        for bx, by in dl:
            loss = F.binary_cross_entropy_with_logits(head(bx).squeeze(-1), by)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return head

def evaluate_head(head, test_data):
    head.eval()
    with torch.inference_mode():
        scores = head(test_data['activations']).squeeze(-1).numpy()
    rho, p = partial_spearman(scores, test_data['losses'],
                               [test_data['max_softmax'], test_data['activation_norm']])
    return scores, rho, p

## Load model and data

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-14B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16,
    attn_implementation='sdpa'
).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f'{sum(p.numel() for p in model.parameters())/1e9:.1f}B params, {N_LAYERS} layers, {HIDDEN_DIM} dim')

TARGET_EX_PER_DIM = 350
MAX_TRAIN = TARGET_EX_PER_DIM * HIDDEN_DIM
MAX_TEST = MAX_TRAIN // 2

wiki_train = load_wikitext('train', max_docs=12000)
wiki_test = load_wikitext('test', max_docs=None)
print(f'{len(wiki_train)} train, {len(wiki_test)} test')

## Confirm peak layer

In [ ]:
wiki_val = load_wikitext('validation', max_docs=None)
candidates = [24, 26, 28, 30, 32, 34]
layer_check = {}

for layer in candidates:
    tr = collect_layer_data(model, tokenizer, wiki_train, layer, 'cuda', MAX_TRAIN)
    va = collect_layer_data(model, tokenizer, wiki_val, layer, 'cuda', MAX_TEST)
    head = train_linear_binary(tr, seed=42)
    _, rho, _ = evaluate_head(head, va)
    layer_check[layer] = float(rho)
    print(f'  layer {layer}: {rho:+.4f} ({len(tr["losses"])} train, {len(tr["losses"])/HIDDEN_DIM:.0f} ex/dim)')
    del tr, va; torch.cuda.empty_cache()

PEAK_LAYER = max(layer_check, key=layer_check.get)
print(f'\nPeak: layer {PEAK_LAYER} ({PEAK_LAYER/N_LAYERS:.0%} depth) = {layer_check[PEAK_LAYER]:+.4f}')

## Collect activations at peak

In [ ]:
train_data = collect_layer_data(model, tokenizer, wiki_train, PEAK_LAYER, 'cuda', MAX_TRAIN)
test_data = collect_layer_data(model, tokenizer, wiki_test, PEAK_LAYER, 'cuda', MAX_TEST)
print(f'Train: {len(train_data["losses"])} ({len(train_data["losses"])/HIDDEN_DIM:.0f} ex/dim)')
print(f'Test: {len(test_data["losses"])} ({len(test_data["losses"])/HIDDEN_DIM:.0f} ex/dim)')

del model; gc.collect(); torch.cuda.empty_cache()
print('Model unloaded.')

## 20-seed hardening

In [ ]:
SEEDS = list(range(42, 62))  # 20 seeds
all_rhos = []
all_scores = []

for seed in SEEDS:
    head = train_linear_binary(train_data, seed=seed)
    scores, rho, p = evaluate_head(head, test_data)
    all_rhos.append(float(rho))
    all_scores.append(scores)
    print(f'  seed {seed}: {rho:+.4f}')

mean_rho = float(np.mean(all_rhos))
std_rho = float(np.std(all_rhos))

# Pairwise seed agreement
pairwise = []
for i in range(len(SEEDS)):
    for j in range(i + 1, len(SEEDS)):
        r, _ = spearmanr(all_scores[i], all_scores[j])
        pairwise.append(float(r))

mean_agree = float(np.mean(pairwise))

# Bootstrap CI
rng = np.random.default_rng(0)
arr = np.array(all_rhos)
boot_means = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(10000)]
ci_lo = float(np.percentile(boot_means, 2.5))
ci_hi = float(np.percentile(boot_means, 97.5))

print(f'\n20-SEED HARDENING')
print(f'  Partial corr: {mean_rho:+.4f} +/- {std_rho:.4f}')
print(f'  95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]')
print(f'  Range: [{min(all_rhos):+.4f}, {max(all_rhos):+.4f}]')
print(f'  Seed agreement: {mean_agree:+.4f}')
print(f'\nFor comparison:')
print(f'  GPT-2 124M (20 seeds): +0.282 +/- 0.001, agreement +0.993')
print(f'  Qwen 7B (3 seeds): +0.240 +/- 0.022, agreement +0.935')

## Save

In [ ]:
output = {
    'model': MODEL_ID,
    'peak_layer': PEAK_LAYER,
    'peak_layer_frac': round(PEAK_LAYER / N_LAYERS, 2),
    'layer_candidates': {str(k): v for k, v in layer_check.items()},
    'n_seeds': len(SEEDS),
    'token_budget': {
        'target_ex_per_dim': TARGET_EX_PER_DIM,
        'train_tokens': int(len(train_data['losses'])),
        'train_ex_per_dim': round(len(train_data['losses']) / HIDDEN_DIM, 1),
    },
    'partial_corr': {
        'mean': mean_rho,
        'std': std_rho,
        'ci_95': [ci_lo, ci_hi],
        'per_seed': all_rhos,
    },
    'seed_agreement': {
        'mean': mean_agree,
        'n_pairs': len(pairwise),
    },
}

with open('qwen14b_hardening.json', 'w') as f:
    json.dump(output, f, indent=2)
print(json.dumps(output, indent=2))

In [ ]:
from google.colab import files
files.download('qwen14b_hardening.json')